# ⚡ **AI FaceSwapper Pro**
A studio-grade, high-accuracy AI face swapping and multi-person face replacement engine powered by InsightFace, InSwapper-128 ONNX & GFPGAN v1.4 HD Restoration.

[GitHub Repository](https://github.com/OddBoyXD/ai-faceswapper-pro) — [Report Issue](https://github.com/OddBoyXD/ai-faceswapper-pro/issues) — [Documentation](https://github.com/OddBoyXD/ai-faceswapper-pro#readme)

---

#### **Highlights & Features**
* 👤 **Single Face Swap**: 1:1 photorealistic facial transfer with expression & lighting adaptation
* 👥 **Multi-Face Group Swap**: Swap 2+ people in group photos with custom target-to-source mapping
* ✨ **GFPGAN v1.4 HD Face Restoration**: Auto crystal-clear upscaling and texture sharpening
* 🚀 **NVIDIA CUDA GPU Acceleration**: Sub-second rendering on free Google Colab T4 GPU
* 🌐 **Instant Public Access**: Gradio Live URL, Localtunnel, or Ngrok sharing with 1-click

#### **Ethical Usage Disclaimer**
This tool is provided for creative, educational, and research purposes only. Please respect privacy, obtain proper consent before using anyone's likeness, and adhere to local laws and ethical standards.

### **1. Install AI FaceSwapper Pro**
Run these cells to mount your Google Drive (optional) and install all required AI models and packages.

In [ ]:
# @title Mount Google Drive (Optional)
# @markdown 💾 Mount your Google Drive to save generated outputs and backups permanently.
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/FaceSwapper_Outputs', exist_ok=True)
    print('✅ Google Drive mounted successfully! Output folder ready: /content/drive/MyDrive/FaceSwapper_Outputs')
except Exception as e:
    print(f'⚠️ Google Drive skipped or not mounted: {e}')

In [ ]:
# @title Setup Runtime Environment & Download AI Models
# @markdown ⚙️ Installs dependencies, sets up ONNX Runtime GPU (CUDA), and downloads InsightFace & InSwapper models.
import os
import sys
from IPython.display import clear_output

%cd /content

print('📦 Step 1/3: Cloning / Updating AI FaceSwapper Pro repository...')
if not os.path.exists('/content/ai-faceswapper-pro'):
    !git clone https://github.com/OddBoyXD/ai-faceswapper-pro.git /content/ai-faceswapper-pro
else:
    %cd /content/ai-faceswapper-pro
    !git pull

%cd /content/ai-faceswapper-pro

print('⚡ Step 2/3: Installing GPU dependencies & UV package manager...')
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx ffmpeg
!pip install -q --upgrade pip
!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!pip install -q fastapi uvicorn gradio insightface opencv-python-headless pillow requests ngrok
!npm install -g -q localtunnel &> /dev/null

print('📥 Step 3/3: Downloading AI Models (InSwapper 128 ONNX + GFPGAN 1.4 HD)...')
!python download_models.py

clear_output()
print('=' * 60)
print('✅ Environment setup complete! Ready to start AI FaceSwapper Pro.')
print('=' * 60)

### **2. Start AI FaceSwapper Pro WebUI**
Select your preferred public sharing tunnel and click Run to get your web link.

In [ ]:
# @title **Start WebUI Server**
# @markdown ### Choose a sharing tunnel method:
from IPython.display import clear_output
import os
import sys

method = "gradio"  # @param ["gradio", "localtunnel", "ngrok"]
ngrok_token = ""  # @param {type:"string"}

%cd /content/ai-faceswapper-pro

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        print(f'🚀 Running on GPU: {torch.cuda.get_device_name(0)}')
    else:
        print('⚠️ No GPU detected. For best speed, go to Runtime -> Change runtime type -> T4 GPU')
except Exception:
    pass

print(f'🌐 Starting AI FaceSwapper Pro via [{method}] tunnel...')

match method:
    case 'gradio':
        !python app.py --share
    case 'localtunnel':
        !echo Password IP: $(curl --silent https://ipv4.icanhazip.com)
        !echo
        !lt --port 7860 & python app.py
    case 'ngrok':
        if not ngrok_token.strip():
            print('❌ Please paste your Ngrok Auth Token from https://dashboard.ngrok.com/get-started/your-authtoken')
        else:
            import ngrok
            ngrok.kill()
            listener = ngrok.forward(7860, authtoken=ngrok_token)
            print(f'🔗 Ngrok Public URL: {listener.url()}')
            !python app.py